# Subclassing numpy.memmap

To circumvent the restriction on `memmap` from the max number of open file, it may be possible to subclass either the `memmap` or the `ndarray` class, so it has a filename, offset, and shape attribute, which are then used when one tries to actually make use of the data.

The relevant reference here is [Subclassing ndarray](https://docs.scipy.org/doc/numpy-1.13.0/user/basics.subclassing.html).

As per the tests below, the definition call to `numpy.memmap`calls `__new__`, while when using the resulting array in an expression, the `__array_prepare__` procedure is called.   This may be used to close the `_mmap` in `__new__`, while saving sufficient data in a new attribute to make it possible to reopen the `_mmap` object in the call to `_array_prepare__`

In [4]:
import numpy as np
import mmap

class memmap2(np.memmap):

    def __new__(subtype, *args, **kwargs):
        obj = super(memmap2, subtype).__new__(subtype,  *args, **kwargs)
        obj._saved = {'len':obj._mmap.__len__(),'off':obj.offset,
                      'file':obj.filename}
        fd=open(obj.filename,'rb+')
        saved=(fd.fileno(),obj._mmap.__len__(),obj.offset)
        obj._mmap.close()
        return obj
    
    def reopen(self):
        s=self._saved
        if self._mmap.closed:
            print('was closed, reopen')
            fd=open(s['file'],'rb+')
            self._mmap=mmap.mmap(fd.fileno(),offset=s['off'],
                                 length=s['len'])
        else:
            print('was open')

    def __getitem__(subtype, index):
        #subtype.reopen()
        obj = super(memmap2, subtype).__getitem__(index)
        return obj

    def __subclasshook__(subtype, *arg):
        subtype.reopen()
        obj = super(memmap2, subtype).__subclasshook__(*arg)
        return obj

    def __array_prepare__ (subtype, *args, **kwargs):
        s=subtype._saved
        if subtype._mmap.closed:
            print('was closed, reopen')
            fd=open(s['file'],'rb+')
            subtype._mmap=mmap.mmap(fd.fileno(),offset=s['off'],
                                    length=s['len'])
        else:
            print('was open')
        obj = super(memmap2, subtype).__array_prepare__(subtype,
                                                        *args, **kwargs)
        return obj

In [5]:
class memmap3(np.memmap):

    def __new__(subtype, file, dtype=np.float32, mode='r+', offset=0, 
                shape=(10,), order='F'):
        self = super(memmap3, subtype).__new__(subtype, file, dtype=dtype, 
                    mode=mode, offset=offset, shape=shape, order=order)
        print('-> memmap3.__new__')
        length = self._mmap.__len__()
        offset = self.offset
        saved = (file,length,offset)
        self._saved = saved
        self._mmap.close()
        return self

    def reopen(self):
        (file,length,offset)=self._saved
        print("-> memmap3.reopen")
        if self._mmap.closed:
            print('--> was closed, reopen')
            fd=open(file,'rb+')
            self._mmap=mmap.mmap(fd.fileno(),length,offset=offset)
        else:
            print('--> was already open')

    def __array_wrap__(self, arr, context=None):
        print('-> memmap3.__array_wrap__')
        arr = super(memmap3, self).__array_wrap__(arr, context)
        return arr

    def __array_prepare__ (self, obj, context=None):
        print('-> memmap3.__array_prepare__')
        self.reopen()
        obj = super(memmap3, self).__array_prepare__(obj, context)
        return obj

    def __array_finalize__ (self, obj):
        print('-> memmap3.__array_finalize__')
        super(memmap3,self).__array_finalize__(obj)
        if obj is None:
            return
        self._saved = obj._saved
        self._mmap = obj._mmap

    def __repr__(self):
        print('-> memmap3.__repr__')
        self.reopen()
        output = super(memmap3,self).__repr__()
        return output

    def __str__(self):
        print('-> memmap3.__str__')
        self.reopen()
        output = super(memmap3,self).__str__()
        return output


Open a `numpy.memmap` for writing, write some values, and show that the file is still open:

In [6]:
mm=np.memmap('tmp.dat',mode='w+',order='F',dtype=np.float32,shape=(10,))
mm[:]=1.0
(mm,mm._mmap.closed)

(memmap([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.], dtype=float32),
 False)

Open for reading, and show the same:

In [7]:
mm=np.memmap('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(10,))
(mm,mm._mmap.closed)

(memmap([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.], dtype=float32),
 False)

Open the `memmap2` subclass for reading, and show that the file is closed

In [8]:
mm2=memmap2('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(10,))
(mm2._saved,mm2._mmap.closed)

({'len': 40,
  'off': 0,
  'file': 'C:\\cygwin64\\home\\codes\\dispatch2_clean\\tests\\python\\memmap\\tmp.dat'},
 True)

In [9]:
mm3=memmap3('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(10,))
(mm3,mm3._saved,mm3._mmap.closed)

-> memmap3.__array_finalize__
-> memmap3.__new__
-> memmap3.__repr__
-> memmap3.reopen
--> was closed, reopen


(memmap3([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.], dtype=float32),
 ('tmp.dat', 40, 0),
 True)

In [12]:
mm3

-> memmap3.__repr__
-> memmap3.reopen
--> was closed, reopen


memmap3([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.], dtype=float32)

In [ ]:
mm2